In [1]:
pip install -e ..

Obtaining file:///home/sagar/winogender_contextuality
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for winogender_contextuality (pyproject.toml) ... done
  Created wheel for winogender_contextuality: filename=winogender_contextuality-0.0.1-py3-none-any.whl size=3635 sha256=8f83f06750999f9d09cc47523a41c6a1871607f6546726d10fea1df95c59f256
  Stored in directory: /tmp/pip-ephem-wheel-cache-xjmcblpt/wheels/f6/b9/38/03ac5a5ccd63b90faa34c1614fd3e708a9c34ab8edb44270e9
Successfully built winogender_contextuality
  Attempting uninstall: winogender_contextuality
    Found existing installation: winogender_contextuality 0.0.1
    Uninstalling winogender_contextuality-0.0.1:
      Successfully uninstalled winogender_contextuality-0.0.1

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run:

In [2]:
import winogender_contextuality as wc
from winogender_contextuality.modeling.ModelProbs import *
from winogender_contextuality.modeling.contextuality import *
import json
import numpy as np
from pathlib import Path
from collections import Counter
from matplotlib import pyplot as plt
from pathlib import Path
import os
from matplotlib import pyplot as plt
import pandas as pd
import ast
import importlib
import torch.nn.functional as F

2025-08-04 14:30:08.537 | INFO     | winogender_contextuality.config:<module>:13 - PROJ_ROOT path is: /home/sagar/winogender_contextuality
2025-08-04 14:30:08.538 | INFO     | winogender_contextuality.config:<module>:17 - DATA_ROOT path is: /data_users1/sagar/winogender_contextuality


2025-08-04 14:30:14.095 | INFO     | winogender_contextuality.modeling.run_local:<module>:9 - torch available: True
2025-08-04 14:30:14.096 | INFO     | winogender_contextuality.modeling.run_local:<module>:10 - 12.6
2025-08-04 14:30:14.139 | INFO     | winogender_contextuality.modeling.run_local:<module>:12 - _CudaDeviceProperties(name='NVIDIA TITAN Xp', major=6, minor=1, total_memory=12188MB, multi_processor_count=30, uuid=69e3ceae-9a8c-607c-5cd7-cde99d6659aa, L2_cache_size=3MB)


In [3]:
DATA_DIR = Path("/data_users1/sagar/winogender_contextuality/data/processed")

In [4]:
LLAMA32_PATH = DATA_DIR / "measurements_Llama-3.2-1B-Instruct_0.5.ndjson"

In [14]:
def load_ndjson(data_path):
    data = []
    with open(data_path, "r") as f:
        for line in f:
            if line.strip():  # avoid empty lines
                data.append(json.loads(line))

    return data

In [15]:
llama32_data = load_ndjson(LLAMA32_PATH)

In [20]:
len(llama32_data)

40844

In [18]:
llama32_data[0]

{'index': 0,
 'context': {'sent_order': [0, 1],
  'pnoun_order': [0, 0],
  'sentence_1': 'The technician told the customer that BLANK could pay with cash.',
  'sentence_2': 'The technician told the customer that BLANK had completed the repair.',
  'pronouns_1': ['he', 'she'],
  'pronouns_2': ['he', 'she']},
 'measurement': {'BLANK1': 'he', 'BLANK2': 'she'},
 'probabilities': [nan, 1.0]}

In [19]:
llama32_forward_data = [d for d in llama32_data if d['context']['sent_order']==[0,1]]

In [21]:
len(llama32_forward_data)

20223

In [35]:
female_pnouns = {'she', 'her', 'hers'}

orientations = [[0,0], [0,1], [1,0], [1,1]]
test0_dict = {tuple(o): {tuple(o): 0 for o in orientations} for o in orientations}
test0_data = [d for d in llama32_forward_data if d['index']==0]

for d in test0_data:
    orientation = d['context']['pnoun_order']
    measurement = d['measurement']
    encoded_m = tuple([val in female_pnouns for val in measurement.values()])
    test0_dict[tuple(orientation)][encoded_m] += 1

    

test0_dict

{(0, 0): {(0, 0): 21, (0, 1): 79, (1, 0): 0, (1, 1): 0},
 (0, 1): {(0, 0): 0, (0, 1): 100, (1, 0): 0, (1, 1): 0},
 (1, 0): {(0, 0): 0, (0, 1): 0, (1, 0): 100, (1, 1): 0},
 (1, 1): {(0, 0): 0, (0, 1): 0, (1, 0): 0, (1, 1): 100}}

In [44]:
test_arr = []
for outcome_dict in test0_dict.values():
    ctx_total = sum(outcome_dict.values())
    ctx_list = []
    for val in outcome_dict.values():
        ctx_list.append(val/ctx_total)

    test_arr.append(ctx_list)
test_arr = np.array(test_arr)
test_arr
        

array([[0.21, 0.79, 0.  , 0.  ],
       [0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 1.  , 0.  ],
       [0.  , 0.  , 0.  , 1.  ]])

In [39]:
# Probably need to output the tokenized prompt from the function, not the list OR input the tokenized prompt here
ms = MeasurementScenario(observations=['template_1', 'template_2'], 
                         measurements=['he_first', 'she_first'],
                         outcomes=[0,1]) # 1 corresponds to selecting the feminine pronoun

In [42]:
ms.get_scenario_matrix() # later depricated
ms.scenario

<xarray.DataArray (context_pair: 4, outcome_pair: 4)> Size: 128B
array([[0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.],
       [0., 0., 0., 0.]])
Coordinates:
  * context_pair  (context_pair) int64 32B 0 1 2 3
  * outcome_pair  (outcome_pair) int64 32B 0 1 2 3

In [45]:
ms.scenario.values = test_arr

In [48]:
check_feasibility(ms)

2025-08-04 15:32:08.043 | INFO     | winogender_contextuality.modeling.contextuality:check_feasibility:117 - Measurement status: 2


(False,
        message: The problem is infeasible. (HiGHS Status 8: model_status is Infeasible; primal_status is None)
        success: False
         status: 2
            fun: None
              x: None
            nit: 0
          lower:  residual: None
                 marginals: None
          upper:  residual: None
                 marginals: None
          eqlin:  residual: None
                 marginals: None
        ineqlin:  residual: None
                 marginals: None)

In [61]:
def pronoun_context_array(
    index, 
    data,
    forward = True
) -> np.ndarray:

    # Constants
    female_pnouns = {'she', 'her', 'hers'}
    orientations = [[0,0], [0,1], [1,0], [1,1]]

    # Sentence order control
    if forward==True:
        sentence_order = [0,1]
    else: 
        sentence_order = [1,0]

    # Loading data and creating empty dictionary
    measurement_data = [d for d in data if (d['index']==index) and (d['context']['sent_order']==sentence_order)]
    measurement_dict = {tuple(o): {tuple(o): 0 for o in orientations} for o in orientations}
    print(f"Checking contextuality on basis of {len(measurement_data)} measurements.")

    # Collecting empirical probabilities
    for d in measurement_data:
        orientation = d['context']['pnoun_order']
        measurement = d['measurement']
        encoded_m = tuple([val in female_pnouns for val in measurement.values()])
        measurement_dict[tuple(orientation)][encoded_m] += 1

    # Converting dictionary of dictionaries to array
    arr_list = []
    for outcome_dict in measurement_dict.values():
        ctx_total = sum(outcome_dict.values())
        ctx_list = []
        for val in outcome_dict.values():
            try:
                ctx_list.append(val/ctx_total)
            except ZeroDivisionError:
                ctx_list.append(0)
    
        arr_list.append(ctx_list)
    arr = np.array(arr_list)
    
    return arr
    

In [64]:
llama32_contextuality = []
for idx in range(61):
    ms_idx = MeasurementScenario(observations=['template_1', 'template_2'], 
                                 measurements=['he_first', 'she_first'],
                                 outcomes=[0,1])
    ms_idx.get_scenario_matrix()
    ms_idx.scenario.values = pronoun_context_array(idx, llama32_data)
    llama32_contextuality.append(check_feasibility(ms_idx)[0])

Checking contextuality on basis of 400 measurements.
2025-08-04 15:47:48.091 | INFO     | winogender_contextuality.modeling.contextuality:check_feasibility:117 - Measurement status: 2
Checking contextuality on basis of 314 measurements.
2025-08-04 15:47:48.223 | INFO     | winogender_contextuality.modeling.contextuality:check_feasibility:117 - Measurement status: 2
Checking contextuality on basis of 379 measurements.
2025-08-04 15:47:48.357 | INFO     | winogender_contextuality.modeling.contextuality:check_feasibility:117 - Measurement status: 2
Checking contextuality on basis of 400 measurements.
2025-08-04 15:47:48.495 | INFO     | winogender_contextuality.modeling.contextuality:check_feasibility:117 - Measurement status: 2
Checking contextuality on basis of 346 measurements.
2025-08-04 15:47:48.636 | INFO     | winogender_contextuality.modeling.contextuality:check_feasibility:117 - Measurement status: 2
Checking contextuality on basis of 393 measurements.
2025-08-04 15:47:48.775 | I

In [65]:
llama32_contextuality

[False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False]

In [73]:
check_feasibility(ms_idx)[1].status

2025-08-06 13:39:21.304 | INFO     | winogender_contextuality.modeling.contextuality:check_feasibility:117 - Measurement status: 2


2

In [74]:
calculate_contextual_fraction_abramsky(ms_idx)

2025-08-06 13:39:52.800 | WARNING  | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:148 - Probabilities don't sum to 4 (sum = 3.0)
2025-08-06 13:39:52.800 | WARNING  | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:149 - This may indicate an issue with the probability distribution
2025-08-06 13:39:52.803 | INFO     | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:180 - Noncontextual fraction: -0.000000
2025-08-06 13:39:52.803 | INFO     | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:181 - Contextual fraction: 1.000000
2025-08-06 13:39:52.804 | INFO     | winogender_contextuality.modeling.contextuality:calculate_contextual_fraction_abramsky:187 - Model is strongly contextual


1.0

## CbD 4-cycle

In [69]:
test_arr

array([[0.21, 0.79, 0.  , 0.  ],
       [0.  , 1.  , 0.  , 0.  ],
       [0.  , 0.  , 1.  , 0.  ],
       [0.  , 0.  , 0.  , 1.  ]])

In [ ]:
cbd